# Orchestrazione iterativa v1 — Planner → MCP Retrieval → Validation → Orchestrator → Answer

**Obiettivo:** estendere **senza modificare** la baseline lineare del notebook 11. La modalità `iterative` può effettuare fino a N retrieval e mettere in pausa la richiesta per un chiarimento. Lo stato è indipendente dal dominio; il dominio MovieLens resta nei profili già presenti. Questo notebook non riguarda il benchmark di Collaborative Filtering avviato in parallelo.

**Percorsi disponibili:**
- `linear`: usa ESATTAMENTE `build_resource_assistant_workflow(...)` preesistente, senza nuove validazioni.
- `iterative`: riusa `ResourcePlannerAgent`, lo stesso `search_resources` MCP, i profili provider e retrieval. Aggiunge un Validator LLM e un Orchestrator LLM con azioni `RETRIEVE`, `CLARIFY`, `FINISH`.

**Limitazioni del prototipo:** non è un agente arbitrario/general-purpose; non modifica i vincoli strutturati tra retrieval successivi, non dispone di un tool `get_resource` aggiuntivo (non esposto dal MCP attuale), non esegue fine-tuning e non rende scientificamente validata la qualità delle decisioni. Il Validator è un giudice semantico LLM, non una ground truth. Per comparazioni sperimentali mantenere stessi modelli/profili e salvare trace.

**Prerequisiti:** file `.py` aggiunto in `src/movie_rag/`, notebook 11 già funzionante, Qdrant locale e collection indicizzata come nel notebook 10, `.env` con API key. Il reranker è **OFF** di default.''

## 0. Import e root del progetto

In [1]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path
from IPython.display import Markdown, display
from dotenv import load_dotenv

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "movie_rag").is_dir():
            return candidate
    raise FileNotFoundError("Avviare Jupyter dalla root del repository (src/movie_rag mancante)")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env", override=False)
print("Repository:", PROJECT_ROOT)


Repository: /Users/danieleraveglia/Documents/uniMi/TESI/progetto


## 1. Parametri — modificare solo questa cella

`ENABLE_RERANK=False` usa il profilo `fast`. Con `True` usa il profilo `quality` configurato lato server. **L’LLM non sceglie il reranker**. Il profilo qualità di ottobre richiede un blocco `reranker` nel JSON; se non esiste, la cella successiva fornisce un errore esplicito. `LLM_PROVIDER` consente `gemini` oppure `openai` (API OpenAI); nel secondo caso installare il pacchetto `openai` e impostare `OPENAI_API_KEY`.

In [2]:
WORKFLOW_MODE = "iterative"   # "linear" o "iterative"
RUN_LIVE = True              # True solo quando Qdrant e API sono pronti
ENABLE_RERANK = False        # False = FAST, True = QUALITY (più lento su CPU/MPS)
LLM_PROVIDER = "gemini"      # "gemini" | "openai"
GEMINI_LLM_PROFILE = "gemini_free"  # nome definito in configs/llm_profiles_v1.json
OPENAI_MODEL = "gpt-4.1-mini"       # adattabile a modello disponibile via API
DOMAIN_ID = "movielens_v2"
QUERY_REWRITE_STRATEGY = "translate_to_catalog_language"
USER_QUERY = "Vorrei film sull'intelligenza artificiale, alcuni filosofici e altri d'azione, non horror."
USER_REPLY = None  # impostare testo dopo un eventuale chiarimento; non chiama input()

MCP_PROFILE = "movielens_gte_quality_v2" if ENABLE_RERANK else "movielens_gte_fast_v2"
LLM_CONFIG_PATH = PROJECT_ROOT / "configs" / "llm_profiles_v1.json"
DOMAIN_CONFIG_PATH = PROJECT_ROOT / "configs" / "resource_domains_v1.json"
MCP_CONFIG_PATH = PROJECT_ROOT / "configs" / "resource_mcp_profiles_v1.json"
TRACE_DIR = PROJECT_ROOT / "outputs" / "orchestration_runs"

MAX_RETRIEVALS = 3
MAX_CLARIFICATIONS = 1
MAX_DECISIONS = 8
MAX_LLM_CALLS = 16
MAX_TOTAL_TOKENS = None  # best-effort; richiede usage fields nel provider
MAX_WALL_SECONDS = 240.0
RESULT_LIMIT = 10

os.environ["RESOURCE_RAG_PROFILE"] = MCP_PROFILE
os.environ.setdefault("QDRANT_URL", "http://localhost:6333")
assert WORKFLOW_MODE in {"linear", "iterative"}
assert LLM_PROVIDER in {"gemini", "openai"}
print({"mode": WORKFLOW_MODE, "provider": LLM_PROVIDER, "rerank": ENABLE_RERANK, "mcp_profile": MCP_PROFILE, "run_live": RUN_LIVE})


{'mode': 'iterative', 'provider': 'gemini', 'rerank': False, 'mcp_profile': 'movielens_gte_fast_v2', 'run_live': True}


## 2. Import del controller (senza dipendenze da Qdrant per i test)

In [3]:
from movie_rag.resource_iterative_orchestration import (
    OrchestrationLimits, ResourceIterativeOrchestrator,
    OpenAITextProvider, normalize_mcp_search_result,
)
limits = OrchestrationLimits(
    max_retrievals=MAX_RETRIEVALS, max_clarifications=MAX_CLARIFICATIONS,
    max_decisions=MAX_DECISIONS, max_llm_calls=MAX_LLM_CALLS,
    max_total_tokens=MAX_TOTAL_TOKENS, max_wall_seconds=MAX_WALL_SECONDS,
    result_limit=RESULT_LIMIT,
)
print("Controller disponibile; nessuna chiamata API")


Controller disponibile; nessuna chiamata API


## 3. Smoke test offline (consigliato prima del live)

Esegue i test unitari nel repository locale senza chiavi API, MCP, modelli o GPU. Sono coperti: più retrieval con merging dei candidati, pause/resume, blocco query duplicate, whitelist degli ID e trace JSON persistenti.

In [4]:
import subprocess
run_tests = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests",
     "-p", "test_resource_iterative_orchestration.py", "-v"],
    cwd=PROJECT_ROOT, text=True, capture_output=True,
    env={**os.environ, "PYTHONPATH": str(SRC_ROOT if "SRC_ROOT" in globals() else PROJECT_ROOT / "src")
         + os.pathsep + os.environ.get("PYTHONPATH", "")},
)
print(run_tests.stdout)
print(run_tests.stderr)
if run_tests.returncode:
    raise RuntimeError("Test offline falliti, correggere prima del live")


[{'resource_id': 'r1', 'title': 'Theory', 'text_excerpt': 'Resource text', 'metadata': {'source': 'test'}, 'queries': ['theory of regression']}]
[{'resource_id': 'r1', 'title': 'Theory', 'text_excerpt': 'Resource text', 'metadata': {'source': 'test'}, 'queries': ['theory of regression']}, {'resource_id': 'r2', 'title': 'Exercises', 'text_excerpt': 'Resource text', 'metadata': {'source': 'test'}, 'queries': ['theory of regression']}]
[{'resource_id': 'r1', 'title': 'Theory', 'text_excerpt': 'Resource text', 'metadata': {'source': 'test'}, 'queries': ['theory of regression']}, {'resource_id': 'r2', 'title': 'Exercises', 'text_excerpt': 'Resource text', 'metadata': {'source': 'test'}, 'queries': ['theory of regression']}]
[{'resource_id': 'r1', 'title': 'Theory', 'text_excerpt': 'Resource text', 'metadata': {'source': 'test'}, 'queries': ['theory of regression']}]
[{'resource_id': 'r1', 'title': 'Theory', 'text_excerpt': 'Resource text', 'metadata': {'source': 'test'}, 'queries': ['theory

## 4. Costruzione dei componenti reali — solo se `RUN_LIVE=True`

Usiamo le **stesse importazioni e factory del notebook 11**. Il profilo MCP resta server-side. La modalità `linear` usa l’implementazione preesistente, senza copiare il suo codice.

In [5]:
if RUN_LIVE:
    from movie_rag.resource_domain import load_resource_domain_catalog
    from movie_rag.resource_mcp_profiles import ResourceSearchServiceRegistry, load_resource_mcp_profiles
    from movie_rag.resource_adapters import movie_filters_to_resource_filters
    from movie_rag.resource_mcp import create_resource_mcp_server
    from movie_rag.resource_llm_agents import ResourcePlannerAgent
    from movie_rag.resource_llm_workflow import build_resource_assistant_workflow
    from movie_rag import GeminiProvider, load_llm_config
    from mcp import Client

    domain = load_resource_domain_catalog(DOMAIN_CONFIG_PATH).get(DOMAIN_ID)
    profile = load_resource_mcp_profiles(MCP_CONFIG_PATH).get(MCP_PROFILE)
    assert profile.domain_id == domain.domain_id
    assert not domain.template_only
    if ENABLE_RERANK and profile.reranker is None:
        raise RuntimeError(
            "Il profilo quality non ha il blocco reranker nel JSON. "
            "Usa la configurazione rifattorizzata dell'8 ottobre o ENABLE_RERANK=False."
        )
    registry = ResourceSearchServiceRegistry.from_environment(MCP_CONFIG_PATH, profile_id=MCP_PROFILE)
    if ENABLE_RERANK and registry.service.reranker is None:
        raise RuntimeError("Reranking richiesto ma non configurato")
    mcp_server = create_resource_mcp_server(
        registry.service, name="Resource retrieval iterative notebook",
        filter_adapter=movie_filters_to_resource_filters,
        default_profile_id=registry.profile.search_profile_id,
        expose_profile_to_client=False, exact=registry.profile.exact,
    )
    if LLM_PROVIDER == "gemini":
        llm_catalog = load_llm_config(LLM_CONFIG_PATH)
        provider = GeminiProvider(llm_catalog.get_llm_profile(GEMINI_LLM_PROFILE))
        model_label = f"gemini:{GEMINI_LLM_PROFILE}"
    else:
        provider = OpenAITextProvider(model=OPENAI_MODEL)
        model_label = f"openai:{OPENAI_MODEL}"
    planner = ResourcePlannerAgent(
        provider=provider, domain=domain,
        temperature=0.0, query_rewrite_strategy=QUERY_REWRITE_STRATEGY,
    )
    print("Profilo MCP:", MCP_PROFILE, "| reranker:", registry.service.reranker.name if registry.service.reranker else "OFF")
else:
    print("RUN_LIVE=False: nessuna connessione Qdrant/API, nessun modello locale caricato")


[10/22/26 11:38:35] INFO     HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"              ]8;id=292362;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=292363;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\

Profilo MCP: movielens_gte_fast_v2 | reranker: OFF


## 5. Confronto lineare / iterativo con lo stesso MCP

I due percorsi sono selezionati da `WORKFLOW_MODE`. In modalità iterativa, il server `search_resources` viene interrogato con `query`, `filters` e `limit`: sono gli stessi argomenti del notebook 11. La risposta MCP viene normalizzata e il controller gestisce un insieme cumulativo dei candidati. In modalità lineare eseguiamo la **factory originale**. La chiamata al tool è esplicita, non affidata liberamente all’LLM.

In [6]:
USER_QUERY = "Vorrei dei film con Tom Hanks prodotti prima del 2010"

async def run_selected_workflow(query: str):
    if not RUN_LIVE:
        raise RuntimeError("Impostare RUN_LIVE=True per eseguire con API/MCP reali")
    async with Client(mcp_server, raise_exceptions=True) as client:
        if WORKFLOW_MODE == "linear":
            from movie_rag.resource_mcp_gateway import MCPResourceToolGateway
            gateway = MCPResourceToolGateway(client)
            workflow = build_resource_assistant_workflow(
                domain=domain, planner_provider=provider, answer_provider=provider,
                gateway=gateway, workflow_id="grounded_resource_v1",
                result_limit=RESULT_LIMIT,
                query_rewrite_strategy=QUERY_REWRITE_STRATEGY,
            )
            return await workflow.run(query)

        async def plan_fn(text: str):
            return await planner.run(text)

        async def search_fn(semantic_query: str, filters: dict, limit: int):
            # FastMCP Client: stesso tool lato server, senza duplicare RetrievalService.
            response = await client.call_tool(
                "search_resources",
                {"query": semantic_query, "filters": filters, "limit": limit},
            )
            return normalize_mcp_search_result(response)

        iterative_workflow = ResourceIterativeOrchestrator(
            planner=plan_fn, search=search_fn, provider=provider,
            domain_instructions=str(getattr(domain, "planner_instructions", "")),
            limits=limits, trace_dir=TRACE_DIR,
            domain_id=domain.domain_id, llm_model_label=model_label,
            retrieval_profile=MCP_PROFILE,
        )
        state = await iterative_workflow.run(query)
        return state

if RUN_LIVE:
    state = await run_selected_workflow(USER_QUERY)
    print("Stato:", state.status)
else:
    state = None
    print("Ready: cambiare RUN_LIVE=True quando il server e le credenziali sono pronti")


[10/22/26 11:38:37] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=292370;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=292371;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

Stato: WAITING_USER


## 6. Ispezione del risultato

Il formato dello stato iterativo è generico: `profile`, `attempts`, `candidates`, `validations`, `events`, `final_answer`. Il formato lineare rimane quello originale (`state.plan`, `state.search_response`, `state.final_answer`).

In [17]:
if RUN_LIVE and state is not None:
    if WORKFLOW_MODE == "linear":
        print("Status:", state.status)
        print("Plan:", state.plan.model_dump() if state.plan else None)
        print("Result count:", state.search_response.returned_count if state.search_response else 0)
        if state.final_answer:
            display(Markdown(state.final_answer.answer))
    else:
        print({"run_id": state.run_id, "status": state.status,
               "retrievals": len(state.attempts), "validations": len(state.validations),
               "candidates": len(state.candidates), "llm_calls": state.llm_calls,
               "token_total_reported": state.token_total,
               "token_usage_complete": state.token_usage_complete,
               "trace": str(TRACE_DIR / state.run_id / "state.json")})
        for i, attempt in enumerate(state.attempts, 1):
            print(f"retrieval #{i}:", attempt.semantic_query, "→", attempt.returned_ids)
        for i, val in enumerate(state.validations, 1):
            print(f"validation #{i}:", val)
        if state.status == "WAITING_USER":
            display(Markdown(f"**Chiarimento richiesto:** {state.pending_question}"))
        elif state.final_answer:
            display(Markdown(state.final_answer["answer"]))
            print("Resource IDs verificati:", state.final_answer["resource_ids"])
else:
    print("Eseguire il notebook in modalità live per vedere la risposta")


{'run_id': 'a49f6f5596cc4d6e', 'status': 'COMPLETE', 'retrievals': 2, 'validations': 2, 'candidates': 20, 'llm_calls': 6, 'token_total_reported': 0, 'token_usage_complete': False, 'trace': '/Users/danieleraveglia/Documents/uniMi/TESI/progetto/outputs/orchestration_runs/a49f6f5596cc4d6e/state.json'}
retrieval #1: Horror, adventure, and comedy movies → ['93840', '4388', '71535', '36401', '122904', '6888', '122916', '4734', '103335', '187593']
retrieval #2: Adventure movie after 2000 → ['59615', '106489', '78105', '6764', '98809', '58293', '6564', '53125', '4254', '4367']
validation #1: {'status': 'PARTIAL', 'covered': ['Horror', 'Commedia', 'Anno successivo al 2000'], 'missing': ["Un film specifico che copra chiaramente il genere 'Avventura' rispettando tutti i vincoli insieme agli altri."], 'invalid_resource_ids': [], 'reason': "L'utente richiede 3 film, uno per ciascuno dei seguenti generi: Horror, Avventura, Commedia, tutti usciti dopo il 2000. Tra i candidati forniti abbiamo ottimi e

Ecco tre film usciti dopo il 2000 che coprono i generi richiesti: 'The Cabin in the Woods' (Horror e Commedia), 'Scary Movie 2' (Commedia) e 'Indiana Jones and the Kingdom of the Crystal Skull' (Avventura).

Resource IDs verificati: ['93840', '4388', '59615']


## 7. Eventuale chiarimento: `resume(state, USER_REPLY)`

Per preservare lo stato serve lo **stesso oggetto `state` nel kernel**. La trace JSON è persistente per analisi e audit, ma questa versione non implementa un restore automatico da disco. Il nuovo messaggio dell’utente aggiorna il profilo tramite il Planner e avvia un nuovo retrieval. Non viene richiesto input interattivo finché `USER_REPLY` è `None`.

In [9]:
print("Status after clarification:", state.status)
if state.status == "WAITING_USER":
    print("In attesa dell'utente:", state.pending_question)

Status after clarification: COMPLETE


In [8]:
USER_REPLY = "anzi no, dei film romantici con Tom Holland prodotti dopo il 2010"
if RUN_LIVE and WORKFLOW_MODE == "iterative" and state is not None and state.status == "WAITING_USER":
    if USER_REPLY is None:
        print("In attesa dell'utente:", state.pending_question)
        print("Impostare USER_REPLY e rieseguire questa cella")
    else:
        async with Client(mcp_server, raise_exceptions=True) as client:
            async def plan_fn(text): return await planner.run(text)
            async def search_fn(q, filters, limit):
                result = await client.call_tool("search_resources", {"query": q, "filters": filters, "limit": limit})
                return normalize_mcp_search_result(result)
            iterative_workflow = ResourceIterativeOrchestrator(
                planner=plan_fn, search=search_fn, provider=provider,
                limits=limits, trace_dir=TRACE_DIR,
                domain_id=domain.domain_id, llm_model_label=model_label,
                retrieval_profile=MCP_PROFILE,
            )
            state = await iterative_workflow.resume(state, USER_REPLY)
        print("Status after clarification:", state.status)
        if state.final_answer: display(Markdown(state.final_answer["answer"]))


[10/22/26 11:38:53] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=292376;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=292377;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

[10/22/26 11:38:58] INFO     No device provided, using mps                                             ]8;id=292384;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py\model.py]8;;\:]8;id=292385;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py#190\190]8;;\

[10/22/26 11:39:00] INFO     Loading SentenceTransformer model from                                   ]8;id=292391;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py\model.py]8;;\:]8;id=292392;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py#1022\1022]8;;\
                             Alibaba-NLP/gte-multilingual-base.                                                    

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[10/22/26 11:39:10] INFO     HTTP Request: POST                                                     ]8;id=292397;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=292398;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_v2_gte_multil                
                             ingual_base_v1/points/query "HTTP/1.1 200 OK"                                         

[{'resource_id': '88163', 'title': 'Crazy, Stupid, Love.', 'text_excerpt': "Title: Crazy, Stupid, Love. genres: Comedy, Drama, Romance Overview: Cal Weaver is living the American dream. He has a good job, a beautiful house, great children and a beautiful wife, named Emily. Cal's seemingly perfect life unravels, however, when he learns that Emily has been unfaithful and wants a divorce. Over 40 and suddenly single, Cal is adrift in the fickle world of dating. Enter, Jacob Palmer, a self-styled player who takes Cal under his wing and teaches him how to be a hit with the ladies. Tagline: This is crazy. This is stupid. This is love. keywords: friendship soulmates marriage crisis midlife crisis babysitter womanizer law school los angeles, california middle school relationship love affair divorcee teenage love playful Directors: Glenn Ficarra, John Requa Writers: Dan Fogelman Composers: Christophe Beck, Nick Urata Main cast: Steve Carell, Ryan Gosling, Julianne Moore, Emma Stone, Lio Tipton 

... (7 outputs left out)

[{'resource_id': '88163', 'title': 'Crazy, Stupid, Love.', 'text_excerpt': "Title: Crazy, Stupid, Love. genres: Comedy, Drama, Romance Overview: Cal Weaver is living the American dream. He has a good job, a beautiful house, great children and a beautiful wife, named Emily. Cal's seemingly perfect life unravels, however, when he learns that Emily has been unfaithful and wants a divorce. Over 40 and suddenly single, Cal is adrift in the fickle world of dating. Enter, Jacob Palmer, a self-styled player who takes Cal under his wing and teaches him how to be a hit with the ladies. Tagline: This is crazy. This is stupid. This is love. keywords: friendship soulmates marriage crisis midlife crisis babysitter womanizer law school los angeles, california middle school relationship love affair divorcee teenage love playful Directors: Glenn Ficarra, John Requa Writers: Dan Fogelman Composers: Christophe Beck, Nick Urata Main cast: Steve Carell, Ryan Gosling, Julianne Moore, Emma Stone, Lio Tipton 

                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=292439;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=292440;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

Status after clarification: COMPLETE


Non risultano risorse sufficientemente conformi ai vincoli verificati.

In [11]:
print(state.final_answer)
print(state.run_id)


{'answer': 'Non risultano risorse sufficientemente conformi ai vincoli verificati.', 'resource_ids': []}
75bb96f340c54be6


## 8. Trace e note per la valutazione

Le trace sono create sotto `outputs/orchestration_runs/<run_id>/state.json` e `metadata.json`. Aggiungere `outputs/orchestration_runs/` al `.gitignore` e **non condividere trace con dati personali dell’utente** senza revisione.

Le metriche di rilevanza (es. NDCG) richiedono qrels indipendenti: il giudizio del Validator non è una ground truth. Per isolare il contributo dell’orchestrazione, confrontare anche i costi e, se possibile, una pipeline con numero di retrieval paragonabile. Il budget di token richiede usage effettiva del provider; dove mancano quei campi, il sistema registra `token_usage_complete=False`, mentre i limiti di chiamate restano effettivi.

**Compatibilità:** il notebook 11, il codice del retrieval e i notebook del benchmark collaborativo non vengono modificati da questa estensione. Se i sorgenti locali sono stati aggiornati dopo l’8 ottobre, verificare il contratto del tool MCP prima dei test live.